# 📊 AI Job Market (2025-2026) Salary & Experience Predictor

Ushbu loyihada AI mehnat bozoridagi mutaxassislar maoshini va ularning tajriba darajasini aniqlovchi **Kaskadli Machine Learning (Cascaded ML Pipeline)** arxitekturasi ishlab chiqilgan.

## 1. Ma'lumotlarni yuklash va dastlabki tahlil

In [ ]:
import pandas as pd
import numpy as np

# Datasetni yuklash
file_name = 'ai_jobs_market_2025_2026.csv'
df = pd.read_csv(file_name)
display(df.head())

In [ ]:
# Ma'lumotlar turlari va statistik tahlil
df.info()
print(df.describe())

## 2. Korrelyatsiya tahlili va Data Leakage xavfini yo'qotish

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols_for_corr = ['years_of_experience', 'annual_salary_usd', 'salary_min_usd',
                       'salary_max_usd', 'ai_salary_premium_pct', 'demand_score',
                       'demand_growth_yoy_pct', 'benefits_score_10', 'posting_year',
                       'posting_month']

correlation_matrix = df[numeric_cols_for_corr].corr()
plt.figure(figsize=(10, 5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korrelyatsiya matritsasi')
plt.show()

## 3. Ma'lumotlarni tozalash, Encoding va Normalizatsiya (Preprocessing)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Outlierlarni olib tashlash
df_filtered = df[df['annual_salary_usd'] <= 400000].copy()

# Data leakage beruvchi ustunlarni drop qilish
cols_to_drop = ['job_id', 'salary_min_usd', 'salary_max_usd', 'salary_tier']
df_clean = df_filtered.drop(columns=cols_to_drop, errors='ignore')

# Kam uchraydigan toifalarni birlashtirish
def keep_top_categories(data, column, top_n=10):
    top_categories = data[column].value_counts().nlargest(top_n).index
    data[column] = data[column].where(data[column].isin(top_categories), 'Other')
    return data

df_clean = keep_top_categories(df_clean, 'job_title', 25)
df_clean = keep_top_categories(df_clean, 'country', 15)
df_clean = keep_top_categories(df_clean, 'city', 15)
df_clean = keep_top_categories(df_clean, 'industry', 15)

# Ko'nikmalarni ajratish
top_skills = ['Python', 'SQL', 'Cloud', 'Leadership', 'Communication', 'Research', 'Agile', 'Statistics', 'Linux', 'Problem Solving', 'PyTorch', 'Git', 'Fine-tuning', 'LLMs', 'Kubernetes']
df_clean['required_skills'] = df_clean['required_skills'].fillna('').astype(str)
for skill in top_skills:
    col_name = 'skill_' + skill.lower().replace('-', '_').replace(' ', '_')
    df_clean[col_name] = df_clean['required_skills'].str.contains(skill, case=False, regex=False).astype(int)
df_clean = df_clean.drop(columns=['required_skills'])

# Targets va One-Hot Encoding
y_reg = np.log1p(df_clean['annual_salary_usd'])
X = df_clean.drop(columns=['annual_salary_usd', 'experience_level'])

categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Normalizatsiya
numeric_cols = ['years_of_experience', 'ai_salary_premium_pct', 'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10', 'posting_year', 'posting_month']
scaler = StandardScaler()
X_encoded[numeric_cols] = scaler.fit_transform(X_encoded[numeric_cols])
print('Preprocessing yakunlandi. X shakli:', X_encoded.shape)

## 4. XGBoost Regressor modeli (Maoshni bashorat qilish)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_encoded, y_reg, test_size=0.2, random_state=42)

xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model.fit(X_train_reg, y_train_reg)

y_test_real = np.expm1(y_test_reg)
y_pred_real = np.expm1(xgb_model.predict(X_test_reg))

print(f'MAE: ${mean_absolute_error(y_test_real, y_pred_real):,.2f}')
print(f'R2 Score: {r2_score(y_test_real, y_pred_real):.2f}')

## 5. Random Forest Classifier (Tajriba darajasini aniqlash - Kaskadli usul)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

y_class_binary = X_encoded['is_senior'].values
X_rf = X_encoded.drop(columns=['is_senior'], errors='ignore').copy()
X_rf['predicted_salary'] = np.expm1(xgb_model.predict(X_encoded))

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_class_binary, test_size=0.2, random_state=42, stratify=y_class_binary)

smote = SMOTE(random_state=42)
X_train_rf_smote, y_train_rf_smote = smote.fit_resample(X_train_rf, y_train_rf)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train_rf_smote, y_train_rf_smote)

y_pred_rf = rf_model.predict(X_test_rf)
print(f'Accuracy: {accuracy_score(y_test_rf, y_pred_rf)*100:.2f}%\n')
print(classification_report(y_test_rf, y_pred_rf))

## 6. Modelni kaskadli sinovdan o'tkazish (Pipeline demo)

In [ ]:
test_samples = X_encoded.sample(n=3, random_state=77)
for i in range(len(test_samples)):
    single_dev = test_samples.iloc[[i]]
    salary_log = xgb_model.predict(single_dev)
    salary_real = np.expm1(salary_log)[0]

    dev_rf = single_dev.drop(columns=['is_senior'], errors='ignore').copy()
    dev_rf['predicted_salary'] = salary_real
    level_pred = rf_model.predict(dev_rf)[0]
    level_name = 'Senior/Lead' if level_pred == 1 else 'Junior/Middle'

    print(f'👨‍💻 Mutaxassis #{i+1}:')
    print(f'  💵 Bashorat qilingan maosh: ${salary_real:,.2f}')
    print(f'  📊 Jori darajasi: {level_name}')
    print("-" * 45)